# Mutation Bias Analysis — 14-Species Dataset

**Species analysed:**
- *Bordetella pertussis*
- *Campylobacter jejuni*
- *Escherichia coli*
- *Haemophilus influenzae*
- *Klebsiella pneumoniae*
- *Listeria monocytogenes*
- *Mycobacterium tuberculosis*
- *Neisseria meningitidis*
- *Pseudomonas aeruginosa*
- *Salmonella* Typhimurium
- *Staphylococcus aureus*
- *Staphylococcus epidermidis*
- *Streptococcus agalactiae*
- *Streptococcus pneumoniae*

**Pipeline overview:**
1. Set up working directory and file paths
2. Build the 4×4 mutation matrix
3. Collapse to 6 complementary substitution pairs
4. Normalise by reference genome nucleotide composition
5. Classify mutations by functional consequence using snpEff annotations (snpEff run in terminal)
6. Trinucleotide context analysis
7. Export CSVs for R modelling and Python figure notebooks

---
## Step 1: Preprocessing pipeline
Raw Parsnp VCF output for each species was obtained from the shared team directory and processed through a three-step filtering pipeline prior to mutation spectrum analysis.

- Singleton filter: SNPs were retained only if they carried FILTER == PASS, were non-multi-allelic, and were present in exactly one genome. This isolates mutations unique to a single lineage, which are least likely to reflect selection.
- Recombination removal: A per-genome sliding window (3 or more SNPs within 2000 bp on the same contig) was used to flag likely recombination-derived clusters. Flagged mutations were zeroed out for the relevant genome.
- Final cleanup: The singleton filter was reapplied to the recombination-masked VCFs to produce the final analysis-ready files.

Filtered VCFs were written to vcfs/cleaned/ and carried forward into all subsequent analysis steps.

In [1]:
# Setup: global paths, species tags, and genome counts
# Defines path constants and species identifiers used throughout the notebook.
# count_genomes() reads the #CHROM header line from each VCF to count genome columns.

import os 
import shutil
from collections import defaultdict

JOHANNA = "/home/jovyan/shared-team/2025-masters-project/people/johanna_genomes"
BASE    = "/home/jovyan/shared-team/2025-masters-project/people/alison"

SPECIES_TAGS = {
    "B. pertussis":     "bpertussis",
    "C. jejuni":        "cjejuni",
    "E. coli":          "ecoli",
    "H. influenzae":    "hinfluenzae",
    "K. pneumoniae":    "kpneumoniae",
    "L. monocytogenes": "lmonocytogenes",
    "M. tuberculosis":  "mtuberculosis",
    "N. meningitidis":  "nmeningitidis",
    "P. aeruginosa":    "paeruginosa",
    "S. typhimurium":   "styphimurium",
    "S. aureus":        "saureus",
    "S. epidermidis":   "sepidermidis",
    "S. agalactiae":    "sagalactiae",
    "S. pneumoniae":    "spneumoniae",
}

RAW_VCF_SOURCES = {
    "B. pertussis":     f"{JOHANNA}/bordetella_pertussis/parsnp_results/parsnp.vcf",
    "C. jejuni":        f"{JOHANNA}/campylobacter_d_jejuni/parsnp_results/parsnp.vcf",
    "E. coli":          f"{JOHANNA}/escherichia_coli/parsnp_results/parsnp.vcf",
    "H. influenzae":    f"{JOHANNA}/haemophilus_influenzae/parsnp_results/parsnp.vcf",
    "K. pneumoniae":    f"{JOHANNA}/klebsiella_pneumoniae/parsnp_results/parsnp.vcf",
    "L. monocytogenes": f"{JOHANNA}/listeria_monocytogenes/parsnp_results/parsnp.vcf",
    "M. tuberculosis":  f"{JOHANNA}/mycobacterium_tuberculosis/parsnp_results/parsnp.vcf",
    "N. meningitidis":  f"{JOHANNA}/neisseria_meningitidis/parsnp_results/parsnp.vcf",
    "P. aeruginosa":    f"{JOHANNA}/pseudomonas_aeruginosa/parsnp_results/parsnp.vcf",
    "S. typhimurium":   f"{JOHANNA}/salmonella_typhimurium/parsnp_results/parsnp.vcf",
    "S. aureus":        f"{JOHANNA}/staphylococcus_aureus/parsnp_results/parsnp.vcf",
    "S. epidermidis":   f"{JOHANNA}/staphylococcus_epidermidis/parsnp_results/parsnp.vcf",
    "S. agalactiae":    f"{JOHANNA}/streptococcus_agalactiae/parsnp_results/parsnp.vcf",
    "S. pneumoniae":    f"{JOHANNA}/streptococcus_pneumoniae/parsnp_results/parsnp.vcf",
}

def count_genomes(vcf_path):
    with open(vcf_path) as f:
        for line in f:
            if line.startswith("#CHROM"):
                return len(line.strip().split("\t")) - 9
    return None

# ── Output directory structure ────────────────────────────────────────────────
# os.makedirs() usage: https://docs.python.org/3/library/os.html#os.makedirs
os.chdir(BASE)

DIR_RAW        = "vcfs/raw"        # copied raw Parsnp VCFs
DIR_SINGLETONS = "vcfs/singletons" # after singleton filter
DIR_ALTERED    = "vcfs/altered"    # after recombination removal
DIR_CLEANED    = "vcfs/cleaned"    # final analysis-ready VCFs

for d in [DIR_RAW, DIR_SINGLETONS, DIR_ALTERED, DIR_CLEANED]:
    os.makedirs(d, exist_ok=True)

print("Directory structure ready:")
for d in [DIR_RAW, DIR_SINGLETONS, DIR_ALTERED, DIR_CLEANED]:
    print(" ", d)

Directory structure ready:
  vcfs/raw
  vcfs/singletons
  vcfs/altered
  vcfs/cleaned


In [2]:
# Copy raw Parsnp VCFs into working directory
# Copies the raw Parsnp output for each species from the shared team directory
# into vcfs/raw/, renamed by species tag for clarity.
# shutil.copy2 usage: https://docs.python.org/3/library/shutil.html#shutil.copy2

raw_vcf_paths = {}
for species, src in RAW_VCF_SOURCES.items():
    tag  = SPECIES_TAGS[species]
    dest = os.path.join(DIR_RAW, f"{tag}_parsnp.vcf")
    shutil.copy2(src, dest)
    raw_vcf_paths[species] = dest
    print(f"Copied {species}: {dest}")

Copied B. pertussis: vcfs/raw/bpertussis_parsnp.vcf
Copied C. jejuni: vcfs/raw/cjejuni_parsnp.vcf
Copied E. coli: vcfs/raw/ecoli_parsnp.vcf
Copied H. influenzae: vcfs/raw/hinfluenzae_parsnp.vcf
Copied K. pneumoniae: vcfs/raw/kpneumoniae_parsnp.vcf
Copied L. monocytogenes: vcfs/raw/lmonocytogenes_parsnp.vcf
Copied M. tuberculosis: vcfs/raw/mtuberculosis_parsnp.vcf
Copied N. meningitidis: vcfs/raw/nmeningitidis_parsnp.vcf
Copied P. aeruginosa: vcfs/raw/paeruginosa_parsnp.vcf
Copied S. typhimurium: vcfs/raw/styphimurium_parsnp.vcf
Copied S. aureus: vcfs/raw/saureus_parsnp.vcf
Copied S. epidermidis: vcfs/raw/sepidermidis_parsnp.vcf
Copied S. agalactiae: vcfs/raw/sagalactiae_parsnp.vcf
Copied S. pneumoniae: vcfs/raw/spneumoniae_parsnp.vcf


In [3]:
# Singleton filter
# Retains only SNP rows that:
#   - have FILTER == PASS
#   - are not multi-allelic (no comma in ALT)
#   - are present in exactly one genome (mutation count across genome columns == 1)
# VCF parsing pattern adapted from: https://www.biostars.org/p/416324/

def count_mutations(row):
    """Count the number of 1s in the genome columns (columns 9 onward)."""
    return sum(int(x) for x in row[9:])

def singleton_filter(input_path, output_path):
    """
    Filter a Parsnp VCF to PASS, non-multi-allelic singletons.
    Writes retained rows (including all header lines) to output_path.
    Returns the number of rows retained.
    """
    retained = 0
    with open(input_path, "r") as inp, open(output_path, "w") as out:
        for line in inp:
            if line.startswith("#"):
                out.write(line)
                continue
            row = line.strip().split("\t")
            if row[6].strip() != "PASS":
                continue
            if "," in row[4]:
                continue
            if count_mutations(row) == 1:
                out.write(line)
                retained += 1
    return retained

singleton_vcf_paths = {}
for species, raw_path in raw_vcf_paths.items():
    tag      = SPECIES_TAGS[species]
    out_path = os.path.join(DIR_SINGLETONS, f"{tag}_singletons.vcf")
    n        = singleton_filter(raw_path, out_path)
    singleton_vcf_paths[species] = out_path
    print(f"{species}: {n} singletons retained → {out_path}")

B. pertussis: 3172 singletons retained → vcfs/singletons/bpertussis_singletons.vcf
C. jejuni: 14356 singletons retained → vcfs/singletons/cjejuni_singletons.vcf
E. coli: 44899 singletons retained → vcfs/singletons/ecoli_singletons.vcf
H. influenzae: 7247 singletons retained → vcfs/singletons/hinfluenzae_singletons.vcf
K. pneumoniae: 87982 singletons retained → vcfs/singletons/kpneumoniae_singletons.vcf
L. monocytogenes: 26087 singletons retained → vcfs/singletons/lmonocytogenes_singletons.vcf
M. tuberculosis: 32546 singletons retained → vcfs/singletons/mtuberculosis_singletons.vcf
N. meningitidis: 23532 singletons retained → vcfs/singletons/nmeningitidis_singletons.vcf
P. aeruginosa: 36707 singletons retained → vcfs/singletons/paeruginosa_singletons.vcf
S. typhimurium: 4250 singletons retained → vcfs/singletons/styphimurium_singletons.vcf
S. aureus: 34739 singletons retained → vcfs/singletons/saureus_singletons.vcf
S. epidermidis: 20804 singletons retained → vcfs/singletons/sepidermidi

In [4]:
# Recombination removal
# Detects recombination-like SNP clusters using a per-genome, per-contig sliding window.
# A region is flagged if >= MIN_SNPS mutations appear within WINDOW_BP base pairs
# in the same genome on the same contig. Flagged mutations are zeroed out for that
# genome. Rows where all genomes are subsequently 0 are dropped.
# Parameters: 3 SNPs within 2000 bp.
#
# Two-pointer sliding window approach adapted from: https://github.com/joshquick/snp_calling_scripts


WINDOW_BP = 2000
MIN_SNPS  = 3

def detect_recombination(vcf_path, window_bp=WINDOW_BP, min_snps=MIN_SNPS):
    """
    Identify (chrom, position, genome) triplets that fall within a
    recombination-like SNP cluster for that genome.
    Returns a deduplicated list of flagged (chrom, pos, genome) tuples.
    """
    headers = []
    data    = []
    with open(vcf_path, "r") as f:
        for line in f:
            if line.startswith("#"):
                headers.append(line.rstrip("\n"))
            else:
                data.append(line.strip().split("\t"))
    col_names = headers[-1].split("\t")
    genomes   = col_names[9:]
    flagged   = []
    for genome in genomes:
        col_idx       = col_names.index(genome)
        snp_positions = defaultdict(list)
        for row in data:
            if row[col_idx] == "1":
                snp_positions[row[0]].append(int(row[1]))
        for chrom, positions in snp_positions.items():
            positions.sort()
            left = 0
            for right in range(len(positions)):
                while positions[right] - positions[left] > window_bp:
                    left += 1
                if right - left + 1 >= min_snps:
                    for i in range(left, right + 1):
                        flagged.append((chrom, positions[i], genome))
    return list(set(map(tuple, flagged)))

def apply_recombination_mask(vcf_path, output_path, flagged):
    """
    Zero out flagged (chrom, pos, genome) entries in the VCF.
    Rows where all genome columns become 0 are dropped.
    Returns (rows_written, mutations_zeroed).
    """
    headers = []
    data    = []
    with open(vcf_path, "r") as f:
        for line in f:
            if line.startswith("#"):
                headers.append(line.rstrip("\n"))
            else:
                data.append(line.strip().split("\t"))
    col_names = headers[-1].split("\t")
    col_idx   = {name: i for i, name in enumerate(col_names)}
    row_lookup = {(row[0], row[1]): i for i, row in enumerate(data)}
    zeroed = 0
    for chrom, pos, genome in flagged:
        key = (chrom, str(pos))
        r   = row_lookup.get(key)
        c   = col_idx.get(genome)
        if r is not None and c is not None and data[r][c] == "1":
            data[r][c] = "0"
            zeroed += 1
    written = 0
    with open(output_path, "w") as out:
        for h in headers:
            out.write(h + "\n")
        for row in data:
            if any(row[9:]):
                out.write("\t".join(row) + "\n")
                written += 1
    return written, zeroed

altered_vcf_paths = {}
for species, sing_path in singleton_vcf_paths.items():
    tag              = SPECIES_TAGS[species]
    out_path         = os.path.join(DIR_ALTERED, f"{tag}_altered.vcf")
    flagged          = detect_recombination(sing_path)
    rows_out, zeroed = apply_recombination_mask(sing_path, out_path, flagged)
    altered_vcf_paths[species] = out_path
    print(f"{species}: {len(flagged)} recombination SNPs flagged, "
          f"{zeroed} mutations zeroed, {rows_out} rows retained → {out_path}")

B. pertussis: 924 recombination SNPs flagged, 924 mutations zeroed, 3172 rows retained → vcfs/altered/bpertussis_altered.vcf
C. jejuni: 8207 recombination SNPs flagged, 8207 mutations zeroed, 14356 rows retained → vcfs/altered/cjejuni_altered.vcf
E. coli: 11963 recombination SNPs flagged, 11963 mutations zeroed, 44899 rows retained → vcfs/altered/ecoli_altered.vcf
H. influenzae: 2788 recombination SNPs flagged, 2788 mutations zeroed, 7247 rows retained → vcfs/altered/hinfluenzae_altered.vcf
K. pneumoniae: 23476 recombination SNPs flagged, 23476 mutations zeroed, 87982 rows retained → vcfs/altered/kpneumoniae_altered.vcf
L. monocytogenes: 14425 recombination SNPs flagged, 14425 mutations zeroed, 26087 rows retained → vcfs/altered/lmonocytogenes_altered.vcf
M. tuberculosis: 1337 recombination SNPs flagged, 1337 mutations zeroed, 32546 rows retained → vcfs/altered/mtuberculosis_altered.vcf
N. meningitidis: 18919 recombination SNPs flagged, 18919 mutations zeroed, 23532 rows retained → vcf

In [5]:
# Final cleanup
# Re-applies the singleton + PASS + non-multi-allelic filter to the
# recombination-masked VCFs to produce the final analysis-ready files.

cleaned_vcf_paths = {}

for species, alt_path in altered_vcf_paths.items():
    tag      = SPECIES_TAGS[species]
    out_path = os.path.join(DIR_CLEANED, f"{tag}_foranalysis.vcf")
    retained = singleton_filter(alt_path, out_path)
    cleaned_vcf_paths[species] = out_path
    print(f"{species}: {retained} SNPs in final analysis VCF → {out_path}")

B. pertussis: 2248 SNPs in final analysis VCF → vcfs/cleaned/bpertussis_foranalysis.vcf
C. jejuni: 6149 SNPs in final analysis VCF → vcfs/cleaned/cjejuni_foranalysis.vcf
E. coli: 32936 SNPs in final analysis VCF → vcfs/cleaned/ecoli_foranalysis.vcf
H. influenzae: 4459 SNPs in final analysis VCF → vcfs/cleaned/hinfluenzae_foranalysis.vcf
K. pneumoniae: 64506 SNPs in final analysis VCF → vcfs/cleaned/kpneumoniae_foranalysis.vcf
L. monocytogenes: 11662 SNPs in final analysis VCF → vcfs/cleaned/lmonocytogenes_foranalysis.vcf
M. tuberculosis: 31209 SNPs in final analysis VCF → vcfs/cleaned/mtuberculosis_foranalysis.vcf
N. meningitidis: 4613 SNPs in final analysis VCF → vcfs/cleaned/nmeningitidis_foranalysis.vcf
P. aeruginosa: 17257 SNPs in final analysis VCF → vcfs/cleaned/paeruginosa_foranalysis.vcf
S. typhimurium: 3652 SNPs in final analysis VCF → vcfs/cleaned/styphimurium_foranalysis.vcf
S. aureus: 21116 SNPs in final analysis VCF → vcfs/cleaned/saureus_foranalysis.vcf
S. epidermidis: 15

In [6]:
# Sense check: SNP counts at each filtering stage

def count_snp_rows(vcf_path):
    """Count non-header rows in a VCF file."""
    n = 0
    with open(vcf_path, "r") as f:
        for line in f:
            if not line.startswith("#"):
                n += 1
    return n


print(f"{'Species':<22} {'Raw':>8} {'Singletons':>12} {'Post-recomb':>13} {'Final':>8}")
print("-" * 67)

for species in RAW_VCF_SOURCES:
    n_raw   = count_snp_rows(raw_vcf_paths[species])
    n_sing  = count_snp_rows(singleton_vcf_paths[species])
    n_alt   = count_snp_rows(altered_vcf_paths[species])
    n_final = count_snp_rows(cleaned_vcf_paths[species])
    print(f"{species:<22} {n_raw:>8} {n_sing:>12} {n_alt:>13} {n_final:>8}")

Species                     Raw   Singletons   Post-recomb    Final
-------------------------------------------------------------------
B. pertussis               4328         3172          3172     2248
C. jejuni                 73704        14356         14356     6149
E. coli                  101704        44899         44899    32936
H. influenzae             65167         7247          7247     4459
K. pneumoniae            284361        87982         87982    64506
L. monocytogenes         119167        26087         26087    11662
M. tuberculosis           47790        32546         32546    31209
N. meningitidis          100500        23532         23532     4613
P. aeruginosa            107029        36707         36707    17257
S. typhimurium             6185         4250          4250     3652
S. aureus                 67223        34739         34739    21116
S. epidermidis            58787        20804         20804    15667
S. agalactiae             17309        12093    

In [7]:
# Use the cleaned VCFs going forward
vcf_files = cleaned_vcf_paths

# Sense-check: number of genomes per species
for species, path in vcf_files.items():
    with open(path, "r") as f:
        for line in f:
            if line.startswith("#CHROM"):
                cols      = line.strip().split("\t")
                n_genomes = len(cols) - 9
                print(f"{species}: {n_genomes} genomes")
                break

B. pertussis: 999 genomes
C. jejuni: 999 genomes
E. coli: 992 genomes
H. influenzae: 1000 genomes
K. pneumoniae: 1001 genomes
L. monocytogenes: 1001 genomes
M. tuberculosis: 1001 genomes
N. meningitidis: 1001 genomes
P. aeruginosa: 1001 genomes
S. typhimurium: 1001 genomes
S. aureus: 1001 genomes
S. epidermidis: 995 genomes
S. agalactiae: 1001 genomes
S. pneumoniae: 1001 genomes


---
## Step 2: Build the 4×4 mutation matrix

Each mutation in the VCF is a single base substitution with a reference base (REF) and an alternative base (ALT). Counting all REF→ALT combinations produces a 4×4 matrix of 12 possible substitution types — the diagonal is always zero as a base cannot mutate to itself. This matrix forms the raw mutation spectrum for each species.

In [8]:
# Counts all REF→ALT substitutions across each species' final VCF.
# Mutation counting pattern adapted from:
#   https://stackoverflow.com/questions/635483/what-is-the-best-way-to-implement-nested-dictionaries-in-python

bases = ["A", "C", "G", "T"]

def build_matrix(vcf_path):
    """Build a 4x4 REF->ALT substitution count matrix from a VCF file."""
    matrix = {ref + "->" + alt: 0 for ref in bases for alt in bases}
    with open(vcf_path, "r") as f:
        for line in f:
            if line.startswith("#"):
                continue
            cols = line.split("\t")
            ref  = cols[3]
            alt  = cols[4]
            if ref in bases and alt in bases:
                matrix[ref + "->" + alt] += 1
    return matrix

matrices = {}
for species, path in vcf_files.items():
    matrices[species] = build_matrix(path)
    print(f"\n--- {species} ---")
    print("Ref\\Alt\t" + "\t".join(bases))
    for ref in bases:
        row = [str(matrices[species][ref + "->" + alt]) for alt in bases]
        print(ref + "\t" + "\t".join(row))


--- B. pertussis ---
Ref\Alt	A	C	G	T
A	0	97	161	23
C	123	0	92	604
G	665	90	0	117
T	28	156	92	0

--- C. jejuni ---
Ref\Alt	A	C	G	T
A	0	103	1100	103
C	128	0	46	1601
G	1621	48	0	134
T	80	1122	63	0

--- E. coli ---
Ref\Alt	A	C	G	T
A	0	959	2346	1060
C	2060	0	598	9536
G	9264	587	0	1991
T	1111	2458	966	0

--- H. influenzae ---
Ref\Alt	A	C	G	T
A	0	125	592	107
C	263	0	61	1117
G	1037	59	0	245
T	106	589	158	0

--- K. pneumoniae ---
Ref\Alt	A	C	G	T
A	0	1399	3623	2199
C	5365	0	1936	17671
G	17648	1950	0	5357
T	2206	3717	1435	0

--- L. monocytogenes ---
Ref\Alt	A	C	G	T
A	0	380	1377	550
C	647	0	194	2608
G	2608	233	0	735
T	590	1379	361	0

--- M. tuberculosis ---
Ref\Alt	A	C	G	T
A	0	1318	2470	277
C	1942	0	1939	7790
G	7622	1893	0	1870
T	281	2590	1217	0

--- N. meningitidis ---
Ref\Alt	A	C	G	T
A	0	88	307	19
C	210	0	46	1635
G	1619	54	0	240
T	24	274	97	0

--- P. aeruginosa ---
Ref\Alt	A	C	G	T
A	0	816	1065	422
C	1149	0	912	4164
G	4277	924	0	1187
T	408	1151	782	0

--- S. typhimurium ---
Ref\Alt	A	C	G	T
A	0	1

---
## Step 3: Collapse to 6 complementary substitution pairs

DNA is double stranded so some substitutions are the same mutation just seen from opposite strands e.g. C→A and G→T. To avoid counting these twice, the 12 options are collapsed into 6 pairs using C or T as the reference base (pyrimidine convention).

In [9]:
# Strand collapse approach follows pyrimidine convention described in:
#   Alexandrov et al. (2013) Signatures of mutational processes in human cancer.
#   Nature, 500, 415–421. https://doi.org/10.1038/nature12477
# Complement dictionary pattern: https://docs.python.org/3/library/stdtypes.html#dict
complement = {"A": "T", "T": "A", "C": "G", "G": "C"}

def collapse_matrix(matrix):
    collapsed = {}
    for ref in ["C", "T"]:
        for alt in bases:
            if ref == alt:
                continue
            key      = ref + "->" + alt
            comp_ref = complement[ref]
            comp_alt = complement[alt]
            collapsed[key] = matrix[ref + "->" + alt] + matrix[comp_ref + "->" + comp_alt]
    return collapsed


collapsed_all = {}
for species, matrix in matrices.items():
    collapsed_all[species] = collapse_matrix(matrix)

for species, collapsed in collapsed_all.items():
    print(f"\n--- {species} ---")
    total = sum(collapsed.values())
    for key, count in collapsed.items():
        print(f"  {key}: {count}")
    print(f"  Total: {total}")


--- B. pertussis ---
  C->A: 240
  C->G: 182
  C->T: 1269
  T->A: 51
  T->C: 317
  T->G: 189
  Total: 2248

--- C. jejuni ---
  C->A: 262
  C->G: 94
  C->T: 3222
  T->A: 183
  T->C: 2222
  T->G: 166
  Total: 6149

--- E. coli ---
  C->A: 4051
  C->G: 1185
  C->T: 18800
  T->A: 2171
  T->C: 4804
  T->G: 1925
  Total: 32936

--- H. influenzae ---
  C->A: 508
  C->G: 120
  C->T: 2154
  T->A: 213
  T->C: 1181
  T->G: 283
  Total: 4459

--- K. pneumoniae ---
  C->A: 10722
  C->G: 3886
  C->T: 35319
  T->A: 4405
  T->C: 7340
  T->G: 2834
  Total: 64506

--- L. monocytogenes ---
  C->A: 1382
  C->G: 427
  C->T: 5216
  T->A: 1140
  T->C: 2756
  T->G: 741
  Total: 11662

--- M. tuberculosis ---
  C->A: 3812
  C->G: 3832
  C->T: 15412
  T->A: 558
  T->C: 5060
  T->G: 2535
  Total: 31209

--- N. meningitidis ---
  C->A: 450
  C->G: 100
  C->T: 3254
  T->A: 43
  T->C: 581
  T->G: 185
  Total: 4613

--- P. aeruginosa ---
  C->A: 2336
  C->G: 1836
  C->T: 8441
  T->A: 830
  T->C: 2216
  T->G: 1598


---
## Step 4: Normalise by reference genome nucleotide composition

Raw substitution counts are influenced by genome composition — a species with higher GC content has more C and G sites available to mutate. To correct for this, each collapsed substitution count is divided by the total number of that base pair type in the reference genome (e.g. C→A/G→T is divided by total C+G bases, since both strands contribute). This gives a normalised mutation rate comparable across species.

In [10]:
# Uses glob pattern matching to find each species' reference FASTA by GCF accession
# in Johanna's shared folder, then copies it into a local references/ directory.
# glob.glob() usage: https://docs.python.org/3/library/glob.html#glob.glob

import glob

GCF_ACCESSIONS = {
    "B. pertussis":     "GCF_026001265.1",
    "C. jejuni":        "GCF_002209065.1",
    "E. coli":          "GCF_949553875.1",
    "H. influenzae":    "GCF_003351605.1",
    "K. pneumoniae":    "GCF_035594225.1",
    "L. monocytogenes": "GCF_045288145.1",
    "M. tuberculosis":  "GCF_977011315.1",
    "N. meningitidis":  "GCF_000191525.1",
    "P. aeruginosa":    "GCF_040429605.1",
    "S. typhimurium":   "GCF_054881025.1",
    "S. aureus":        "GCF_900017775.1",
    "S. epidermidis":   "GCF_000011925.1",
    "S. agalactiae":    "GCF_015221735.2",
    "S. pneumoniae":    "GCF_022068225.1",
}

FOLDER_NAMES = {
    "B. pertussis":     "bordetella_pertussis",
    "C. jejuni":        "campylobacter_d_jejuni",
    "E. coli":          "escherichia_coli",
    "H. influenzae":    "haemophilus_influenzae",
    "K. pneumoniae":    "klebsiella_pneumoniae",
    "L. monocytogenes": "listeria_monocytogenes",
    "M. tuberculosis":  "mycobacterium_tuberculosis",
    "N. meningitidis":  "neisseria_meningitidis",
    "P. aeruginosa":    "pseudomonas_aeruginosa",
    "S. typhimurium":   "salmonella_typhimurium",
    "S. aureus":        "staphylococcus_aureus",
    "S. epidermidis":   "staphylococcus_epidermidis",
    "S. agalactiae":    "streptococcus_agalactiae",
    "S. pneumoniae":    "streptococcus_pneumoniae",
}

REFERENCE_SOURCES = {}
for species, gcf in GCF_ACCESSIONS.items():
    pattern = f"{JOHANNA}/{FOLDER_NAMES[species]}/reference_fasta/{gcf}*.fna"
    matches = glob.glob(pattern)
    if len(matches) == 0:
        raise FileNotFoundError(f"No FASTA found for {species} — pattern tried: {pattern}")
    if len(matches) > 1:
        raise ValueError(f"Multiple FASTAs found for {species}: {matches}")
    REFERENCE_SOURCES[species] = matches[0]
    print(f"{species}: {os.path.basename(matches[0])}")
    
os.makedirs("references", exist_ok=True)

fasta_files = {}
for species, src in REFERENCE_SOURCES.items():
    tag  = SPECIES_TAGS[species]
    dest = os.path.join("references", f"{tag}_reference.fna")
    shutil.copy2(src, dest)
    fasta_files[species] = dest
    print(f"Copied {species}: {dest}")

B. pertussis: GCF_026001265.1_ASM2600126v1_genomic.fna
C. jejuni: GCF_002209065.1_ASM220906v1_genomic.fna
E. coli: GCF_949553875.1_sample2_Strain154_Phi80.fasta_genomic.fna
H. influenzae: GCF_003351605.1_ASM335160v1_genomic.fna
K. pneumoniae: GCF_035594225.1_ASM3559422v1_genomic.fna
L. monocytogenes: GCF_045288145.1_ATCC_BAA-679_202309_genomic.fna
M. tuberculosis: GCF_977011315.1_K40X0395_genomic.fna
N. meningitidis: GCF_000191525.1_ASM19152v1_genomic.fna
P. aeruginosa: GCF_040429605.1_ASM4042960v1_genomic.fna
S. typhimurium: GCF_054881025.1_ASM5488102v1_genomic.fna
S. aureus: GCF_900017775.1_NZAK3_genomic.fna
S. epidermidis: GCF_000011925.1_ASM1192v1_genomic.fna
S. agalactiae: GCF_015221735.2_ASM1522173v2_genomic.fna
S. pneumoniae: GCF_022068225.1_ASM2206822v1_genomic.fna
Copied B. pertussis: references/bpertussis_reference.fna
Copied C. jejuni: references/cjejuni_reference.fna
Copied E. coli: references/ecoli_reference.fna
Copied H. influenzae: references/hinfluenzae_reference.fna
Co

In [11]:
# Count reference genome nucleotide composition and normalise mutation rates
# get_base_counts() reads a FASTA file and counts A/C/G/T across all sequence lines.
# normalise() divides each collapsed substitution count by the total number of
# available sites (both strands), giving a rate comparable across species.
# FASTA nucleotide counting pattern adapted from: https://www.biostars.org/p/9462180/
# Normalisation approach follows: Alexandrov et al. (2013) Nature, 500, 415-421.
#   https://doi.org/10.1038/nature12477

def get_base_counts(fasta_path):
    """Count A, C, G, T occurrences across all sequence lines in a FASTA file."""
    base_counts = {"A": 0, "C": 0, "G": 0, "T": 0}
    with open(fasta_path, "r") as f:
        for line in f:
            if line.startswith(">"):
                continue
            for base in line.strip().upper():
                if base in base_counts:
                    base_counts[base] += 1
    return base_counts

def normalise(collapsed, base_counts):
    """
    Divide each substitution count by the total number of available sites
    on both strands (e.g. C→A divided by total C + G bases).
    """
    normalised = {}
    comp       = {"C": "G", "T": "A"}
    for key, count in collapsed.items():
        ref   = key[0]
        denom = base_counts[ref] + base_counts[comp[ref]]
        normalised[key] = count / denom
    return normalised

base_counts_all = {}
normalised_all  = {}
for species in vcf_files:
    bc = get_base_counts(fasta_files[species])
    base_counts_all[species] = bc
    total = sum(bc.values())
    gc    = (bc["G"] + bc["C"]) / total * 100
    print(f"{species} — GC content: {gc:.1f}%")
    normalised_all[species] = normalise(collapsed_all[species], bc)

for species, normalised in normalised_all.items():
    print(f"\n--- {species} ---")
    for key, rate in normalised.items():
        print(f"  {key}: {rate:.6f}")

B. pertussis — GC content: 67.7%
C. jejuni — GC content: 30.6%
E. coli — GC content: 50.8%
H. influenzae — GC content: 38.0%
K. pneumoniae — GC content: 57.3%
L. monocytogenes — GC content: 38.0%
M. tuberculosis — GC content: 65.6%
N. meningitidis — GC content: 51.3%
P. aeruginosa — GC content: 65.6%
S. typhimurium — GC content: 52.2%
S. aureus — GC content: 32.8%
S. epidermidis — GC content: 32.1%
S. agalactiae — GC content: 35.5%
S. pneumoniae — GC content: 39.6%

--- B. pertussis ---
  C->A: 0.000086
  C->G: 0.000065
  C->T: 0.000454
  T->A: 0.000038
  T->C: 0.000238
  T->G: 0.000142

--- C. jejuni ---
  C->A: 0.000529
  C->G: 0.000190
  C->T: 0.006508
  T->A: 0.000163
  T->C: 0.001974
  T->G: 0.000148

--- E. coli ---
  C->A: 0.001679
  C->G: 0.000491
  C->T: 0.007792
  T->A: 0.000930
  T->C: 0.002058
  T->G: 0.000825

--- H. influenzae ---
  C->A: 0.000740
  C->G: 0.000175
  C->T: 0.003137
  T->A: 0.000190
  T->C: 0.001056
  T->G: 0.000253

--- K. pneumoniae ---
  C->A: 0.003415
 

---
## Step 5: Classify mutations by functional consequence

Each SNP in the cleaned VCFs was annotated using snpEff with a custom database built from the reference FASTA and GFF annotation for each species. Mutations were classified into three categories based on the primary effect in the ANN field:

- **Synonymous** — coding variant that does not change the amino acid sequence
- **Non-synonymous** — coding variant that changes the amino acid sequence (missense)
- **Intergenic** — falls outside any annotated gene

The mutation matrix, collapsing, and normalisation steps are repeated separately for each category. Comparing spectra across categories allows assessment of whether selection is distorting the observed mutation pattern in coding regions — synonymous and intergenic mutations are largely neutral and most reflective of the underlying mutation bias.

In [12]:
# Classify SNPs by functional consequence using snpEff annotations
# Parses the ANN field in snpEff-annotated VCFs to classify each SNP as
# synonymous, nonsynonymous, or intergenic based on the primary effect.
# Repeats the matrix, collapse, and normalisation steps for each category.
# snpEff ANN field format: https://pcingola.github.io/SnpEff/snpeff/inputoutput/#ann-field-vcf-output-files
# VCF INFO field parsing pattern adapted from: https://www.biostars.org/p/416324/

snpeff_files = {
    "B. pertussis":     "snpeff/bpertussis_annotated.vcf",
    "C. jejuni":        "snpeff/cjejuni_annotated.vcf",
    "E. coli":          "snpeff/ecoli_annotated.vcf",
    "H. influenzae":    "snpeff/hinfluenzae_annotated.vcf",
    "K. pneumoniae":    "snpeff/kpneumoniae_annotated.vcf",
    "L. monocytogenes": "snpeff/lmonocytogenes_annotated.vcf",
    "M. tuberculosis":  "snpeff/mtuberculosis_annotated.vcf",
    "N. meningitidis":  "snpeff/nmeningitidis_annotated.vcf",
    "P. aeruginosa":    "snpeff/paeruginosa_annotated.vcf",
    "S. typhimurium":   "snpeff/styphimurium_annotated.vcf",
    "S. aureus":        "snpeff/saureus_annotated.vcf",
    "S. epidermidis":   "snpeff/sepidermidis_annotated.vcf",
    "S. agalactiae":    "snpeff/sagalactiae_annotated.vcf",
    "S. pneumoniae":    "snpeff/spneumoniae_annotated.vcf",
}

SNPEFF_SYNONYMOUS = "synonymous_variant"
CODING_EFFECTS = {
    "missense_variant",
    "stop_gained",
    "stop_lost",
    "start_lost",
    "splice_region_variant",
    "initiator_codon_variant",
    "stop_lost&splice_region_variant",
    "splice_region_variant&stop_retained_variant",
}

categories = ["synonymous", "nonsynonymous", "intergenic"]


def get_primary_effect(info_field):
    """Extract the primary effect type from the snpEff ANN field."""
    for part in info_field.split(";"):
        if part.startswith("ANN="):
            primary = part[4:].split(",")[0]
            fields  = primary.split("|")
            if len(fields) > 1:
                return fields[1]
    return None


def classify_vcf(vcf_path):
    """
    Split an annotated VCF into synonymous, nonsynonymous, and intergenic
    categories based on the primary snpEff effect.
    """
    headers    = []
    classified = {cat: [] for cat in categories}

    with open(vcf_path, "r") as f:
        for line in f:
            if line.startswith("#"):
                headers.append(line)
                continue
            cols   = line.strip().split("\t")
            effect = get_primary_effect(cols[7])
            if effect == SNPEFF_SYNONYMOUS:
                classified["synonymous"].append(line)
            elif effect in CODING_EFFECTS:
                classified["nonsynonymous"].append(line)
            else:
                classified["intergenic"].append(line)

    return headers, classified


category_results = {}

for species, vcf_path in snpeff_files.items():
    tag    = SPECIES_TAGS[species]
    folder = os.path.join("snpeff", tag)
    os.makedirs(folder, exist_ok=True)

    headers, classified = classify_vcf(vcf_path)
    shutil.copy2(vcf_path, os.path.join(folder, f"{tag}.ann.vcf"))

    category_results[species] = {}

    for category in categories:
        lines  = classified[category]
        matrix = {ref + "->" + alt: 0 for ref in bases for alt in bases}
        for line in lines:
            cols = line.strip().split("\t")
            ref  = cols[3]
            alt  = cols[4]
            if ref in bases and alt in bases:
                matrix[ref + "->" + alt] += 1

        collapsed  = collapse_matrix(matrix)
        normalised = normalise(collapsed, base_counts_all[species])
        total      = sum(collapsed.values())

        category_results[species][category] = {
            "collapsed":  collapsed,
            "normalised": normalised,
            "total":      total,
        }

        out_path = os.path.join(folder, f"{tag}_{category}.vcf")
        with open(out_path, "w") as out:
            for h in headers:
                out.write(h)
            for line in lines:
                out.write(line)

    print(f"{species}: done")


# Sense check: SNP counts per category
print(f"\n{'Species':<22} {'Synonymous':>12} {'Nonsynonymous':>15} {'Intergenic':>12} {'Total':>8}")
print("-" * 74)
for species in snpeff_files:
    syn    = category_results[species]["synonymous"]["total"]
    nonsyn = category_results[species]["nonsynonymous"]["total"]
    igr    = category_results[species]["intergenic"]["total"]
    print(f"{species:<22} {syn:>12} {nonsyn:>15} {igr:>12} {syn+nonsyn+igr:>8}")

B. pertussis: done
C. jejuni: done
E. coli: done
H. influenzae: done
K. pneumoniae: done
L. monocytogenes: done
M. tuberculosis: done
N. meningitidis: done
P. aeruginosa: done
S. typhimurium: done
S. aureus: done
S. epidermidis: done
S. agalactiae: done
S. pneumoniae: done

Species                  Synonymous   Nonsynonymous   Intergenic    Total
--------------------------------------------------------------------------
B. pertussis                    591            1217          440     2248
C. jejuni                      2182            3884           83     6149
E. coli                       13218           16146         3572    32936
H. influenzae                  1644            2595          220     4459
K. pneumoniae                 28542           27962         8002    64506
L. monocytogenes               4419            6353          890    11662
M. tuberculosis               10507           17171         3531    31209
N. meningitidis                1756            2313       

In [13]:
# Export mutation data to CSV for statistical modelling in R
# Builds one row per species × mutation class × category combination.
# 'opp' is the normalisation denominator (total sites on both strands).
# Model formula used downstream: n_mut ~ mut_class + species * is_igr + offset(log(opp))
# csv.DictWriter usage: https://docs.python.org/3/library/csv.html#csv.DictWriter

import csv

rows = []
for species in snpeff_files:
    for category in categories:
        collapsed = category_results[species][category]["collapsed"]
        bc        = base_counts_all[species]
        for mut_class, n_mut in collapsed.items():
            ref      = mut_class[0]
            comp_ref = {"C": "G", "T": "A"}[ref]
            opp      = bc[ref] + bc[comp_ref]
            rows.append({
                "n_mut":     n_mut,
                "mut_class": mut_class,
                "species":   species,
                "category":  category,
                "opp":       opp,
            })

with open("mutation_data_14sp.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["n_mut", "mut_class", "species", "category", "opp"])
    writer.writeheader()
    writer.writerows(rows)

print(f"Written {len(rows)} rows to mutation_data_14sp.csv")

Written 252 rows to mutation_data_14sp.csv


In [14]:
# Cell 14 — Category-specific opportunity calculation
# Codon enumeration logic adapted from Maisie's implementation.
#
# Synonymous/nonsynonymous opportunity: enumerates all 9 possible single-nucleotide
# changes per codon position across full transcript sequences. CDS blocks are grouped
# by transcript ID to preserve reading frame across multi-exon features.
# Intergenic opportunity: counts C and T bases at non-CDS genomic positions.
#
# GFF3 parsing pattern adapted from: https://techoverflow.net/2013/11/30/a-simple-gff3-parser-in-python/
# Reverse complement using str.maketrans: https://docs.python.org/3/library/stdtypes.html#str.translate
# FASTA nucleotide counting pattern adapted from: https://gist.github.com/katyanna/8a15a1cd8fe3044551de7f1ed9901d7a
# Codon table and synonymous/nonsynonymous enumeration approach follows:
#   Nei, M. and Gojobori, T. (1986) Mol. Biol. Evol., 3(5), 418-426.

import os
import glob
from collections import defaultdict

CODON_TABLE = {
    "TTT":"F","TTC":"F","TTA":"L","TTG":"L",
    "TCT":"S","TCC":"S","TCA":"S","TCG":"S",
    "TAT":"Y","TAC":"Y","TAA":"*","TAG":"*",
    "TGT":"C","TGC":"C","TGA":"*","TGG":"W",
    "CTT":"L","CTC":"L","CTA":"L","CTG":"L",
    "CCT":"P","CCC":"P","CCA":"P","CCG":"P",
    "CAT":"H","CAC":"H","CAA":"Q","CAG":"Q",
    "CGT":"R","CGC":"R","CGA":"R","CGG":"R",
    "ATT":"I","ATC":"I","ATA":"I","ATG":"M",
    "ACT":"T","ACC":"T","ACA":"T","ACG":"T",
    "AAT":"N","AAC":"N","AAA":"K","AAG":"K",
    "AGT":"S","AGC":"S","AGA":"R","AGG":"R",
    "GTT":"V","GTC":"V","GTA":"V","GTG":"V",
    "GCT":"A","GCC":"A","GCA":"A","GCG":"A",
    "GAT":"D","GAC":"D","GAA":"E","GAG":"E",
    "GGT":"G","GGC":"G","GGA":"G","GGG":"G",
}
VALID_BASES = {"A", "C", "G", "T"}
COMPLEMENT  = str.maketrans("ACGTacgt", "TGCAtgca")

def reverse_complement(seq):
    """Return the reverse complement of a DNA sequence."""
    return seq.translate(COMPLEMENT)[::-1]

def parse_fasta_dict(fasta_path):
    """Read a FASTA file and return a dict of {seq_id: sequence}."""
    seqs = {}
    seq_id, chunks = None, []
    with open(fasta_path) as f:
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                if seq_id:
                    seqs[seq_id] = "".join(chunks).upper()
                seq_id = line[1:].split()[0]
                chunks = []
            else:
                chunks.append(line)
    if seq_id:
        seqs[seq_id] = "".join(chunks).upper()
    return seqs

def parse_gff_cds_by_transcript(gff_path):
    """
    Parse GFF and group CDS records by transcript/parent ID.
    Returns dict: transcript_id -> list of CDS block dicts.
    """
    cds_by_transcript = defaultdict(list)
    with open(gff_path) as f:
        for line in f:
            if not line or line.startswith("#"):
                continue
            fields = line.strip().split("\t")
            if len(fields) != 9 or fields[2] != "CDS":
                continue
            seqid, _, _, start, end, _, strand, _, attributes = fields

            attr_map = {}
            for item in attributes.split(";"):
                if "=" in item:
                    k, v = item.split("=", 1)
                    attr_map[k.strip()] = v.strip()

            transcript_id = (
                attr_map.get("Parent") or
                attr_map.get("transcript_id") or
                attr_map.get("ID")
            )
            if transcript_id is None:
                continue

            cds_by_transcript[transcript_id].append({
                "seqid":  seqid,
                "start":  int(start),  # 1-based GFF
                "end":    int(end),
                "strand": strand,
            })
    return cds_by_transcript

def build_transcript_sequences(genome, cds_by_transcript):
    """
    Concatenate CDS blocks per transcript into full coding sequences,
    respecting strand and sorting blocks in correct order.
    Returns dict: transcript_id -> coding sequence string.
    """
    transcript_seqs = {}
    for transcript_id, blocks in cds_by_transcript.items():
        seqid  = blocks[0]["seqid"]
        strand = blocks[0]["strand"]
        if seqid not in genome:
            continue

        chrom = genome[seqid]
        if strand == "+":
            sorted_blocks = sorted(blocks, key=lambda b: b["start"])
            seq = "".join(chrom[b["start"]-1 : b["end"]] for b in sorted_blocks)
        else:
            sorted_blocks = sorted(blocks, key=lambda b: b["start"], reverse=True)
            seq = "".join(
                reverse_complement(chrom[b["start"]-1 : b["end"]])
                for b in sorted_blocks
            )

        seq = seq.upper()
        usable = len(seq) - (len(seq) % 3)
        if usable >= 3:
            transcript_seqs[transcript_id] = seq[:usable]

    return transcript_seqs

def enumerate_codon_opportunities(transcript_seqs):
    """
    For every codon in every transcript, enumerate all 9 possible
    single-nucleotide changes (all 4 bases, all 3 positions).
    Count how many result in synonymous vs nonsynonymous amino acid changes.
    Returns total syn_opp and nonsyn_opp as integers.
    """
    syn_opp    = 0
    nonsyn_opp = 0

    for seq in transcript_seqs.values():
        for i in range(0, len(seq), 3):
            codon = seq[i:i+3]
            if len(codon) != 3 or set(codon) - VALID_BASES:
                continue
            if codon not in CODON_TABLE:
                continue
            aa_ref = CODON_TABLE[codon]

            for pos in range(3):
                ref_base = codon[pos]
                for alt_base in VALID_BASES:
                    if alt_base == ref_base:
                        continue
                    alt_codon = list(codon)
                    alt_codon[pos] = alt_base
                    alt_codon = "".join(alt_codon)
                    if alt_codon not in CODON_TABLE:
                        continue
                    aa_alt = CODON_TABLE[alt_codon]
                    if aa_alt == aa_ref:
                        syn_opp += 1
                    else:
                        nonsyn_opp += 1

    return syn_opp, nonsyn_opp

def intergenic_base_counts(cds_by_transcript, genome):
    """
    Count C and T bases at positions not covered by any CDS feature.
    """
    coding_positions = defaultdict(set)
    for blocks in cds_by_transcript.values():
        for b in blocks:
            for pos in range(b["start"] - 1, b["end"]):  # convert to 0-based
                coding_positions[b["seqid"]].add(pos)

    igr_counts = {"C": 0, "T": 0}
    for seqid, seq in genome.items():
        for pos, base in enumerate(seq):
            if pos not in coding_positions[seqid] and base in igr_counts:
                igr_counts[base] += 1

    return igr_counts


# ── Run for all 14 species ────────────────────────────────────────────────────
category_opp = {}

for species in snpeff_files:
    gff_path = f"{JOHANNA}/{FOLDER_NAMES[species]}/genomic.gff"
    if not os.path.exists(gff_path):
        print(f"WARNING: no GFF found for {species}: {gff_path}")
        continue

    genome            = parse_fasta_dict(fasta_files[species])
    cds_by_transcript = parse_gff_cds_by_transcript(gff_path)
    transcript_seqs   = build_transcript_sequences(genome, cds_by_transcript)
    syn_o, nonsyn_o   = enumerate_codon_opportunities(transcript_seqs)
    igr_o             = intergenic_base_counts(cds_by_transcript, genome)

    category_opp[species] = {
        "synonymous":    syn_o,
        "nonsynonymous": nonsyn_o,
        "intergenic":    igr_o,
    }

    print(f"{species}: syn={syn_o:,}, nonsyn={nonsyn_o:,}, "
          f"igr_C={igr_o['C']:,}, igr_T={igr_o['T']:,}")

B. pertussis: syn=2,939,979, nonsyn=8,430,846, igr_C=107,543, igr_T=67,030
C. jejuni: syn=921,017, nonsyn=3,663,358, igr_C=13,028, igr_T=36,765
E. coli: syn=2,950,549, nonsyn=9,602,264, igr_C=124,658, igr_T=162,047
H. influenzae: syn=1,045,370, nonsyn=3,745,762, igr_C=34,103, igr_T=69,542
K. pneumoniae: syn=3,536,595, nonsyn=11,008,755, igr_C=154,184, igr_T=166,001
L. monocytogenes: syn=1,726,105, nonsyn=6,173,438, igr_C=54,173, igr_T=103,564
M. tuberculosis: syn=3,139,066, nonsyn=8,855,918, igr_C=119,755, igr_T=71,595
N. meningitidis: syn=1,295,218, nonsyn=4,320,737, igr_C=83,241, igr_T=106,854
P. aeruginosa: syn=4,778,201, nonsyn=14,225,389, igr_C=219,633, igr_T=147,474
S. typhimurium: syn=3,232,891, nonsyn=10,430,567, igr_C=138,271, igr_T=174,942
S. aureus: syn=1,514,117, nonsyn=5,733,646, igr_C=65,057, igr_T=157,929
S. epidermidis: syn=1,381,686, nonsyn=5,251,206, igr_C=62,240, igr_T=157,930
S. agalactiae: syn=1,198,905, nonsyn=4,358,577, igr_C=37,466, igr_T=82,544
S. pneumoniae: s

In [15]:
# Export per-category SNP counts and opportunities to CSV
# Aggregates total mutation counts and opportunity denominators per species per category.
# opp is an integer for synonymous/nonsynonymous (codon enumeration total)
# and a dict for intergenic (C + T base counts at non-CDS positions).
# csv.DictWriter usage: https://docs.python.org/3/library/csv.html#csv.DictWriter

import csv

rows = []
for species in snpeff_files:
    if species not in category_opp:
        continue
    for category in categories:
        collapsed = category_results[species][category]["collapsed"]
        n_total   = sum(collapsed.values())
        opp       = category_opp[species][category]
        if category == "intergenic":
            opp_total = opp["C"] + opp["T"]
        else:
            opp_total = opp
        rows.append({
            "species":  species,
            "category": category,
            "n_mut":    n_total,
            "opp":      opp_total,
        })

with open("mutation_data_category_opp.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["species", "category", "n_mut", "opp"])
    writer.writeheader()
    writer.writerows(rows)

print(f"Written {len(rows)} rows to mutation_data_category_opp.csv")

Written 42 rows to mutation_data_category_opp.csv


---
## Step 6: Trinucleotide context analysis

Extends the 6-class spectrum to 96 contexts by incorporating the immediate
5' and 3' flanking bases at each mutation site. Mutation counts are expressed
as a proportion of total mutations per species (COSMIC convention).
Results exported to trinucleotide_data_14sp.csv.


In [16]:
# Trinucleotide context analysis
# Counts observed mutations across all 96 trinucleotide contexts and normalises
# as proportion of total mutations per species (COSMIC convention).
#
# 96-context ordering follows COSMIC pyrimidine convention:
#   Alexandrov et al. (2013) Nature, 500, 415-421. https://doi.org/10.1038/nature12477
# Trinucleotide context generation approach:
#   https://bbglab.github.io/bbgwiki/Methods/Signature/TrinucleotideOrdering/
# Flanking base extraction from reference FASTA adapted from:
#   https://www.biostars.org/p/334253/
# FASTA parsing into dict adapted from:
#   https://gist.github.com/katyanna/8a15a1cd8fe3044551de7f1ed9901d7a
# csv.DictWriter usage: https://docs.python.org/3/library/csv.html#csv.DictWriter

def load_fasta(fasta_file):
    """Load a FASTA file into a dictionary of {contig_name: sequence}."""
    sequences    = {}
    current_name = None
    current_seq  = []
    with open(fasta_file, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if current_name is not None:
                    sequences[current_name] = "".join(current_seq).upper()
                current_name = line[1:].split()[0]
                current_seq  = []
            else:
                current_seq.append(line)
        if current_name is not None:
            sequences[current_name] = "".join(current_seq).upper()
    return sequences


def build_96_context_dictionary():
    """Build an ordered dictionary of all 96 trinucleotide contexts initialised to 0."""
    mutation_types = ["C>A", "C>G", "C>T", "T>A", "T>C", "T>G"]
    context_dict   = {}
    for mut in mutation_types:
        for left in bases:
            for right in bases:
                context_dict[f"{left}[{mut}]{right}"] = 0
    return context_dict


def count_reference_trinucleotide_contexts(fasta_file):
    """
    Count trinucleotide context opportunities in the reference genome.
    Uses pyrimidine convention — A and G centred contexts are reverse complemented.
    """
    sequences   = load_fasta(fasta_file)
    comp        = {"A": "T", "T": "A", "C": "G", "G": "C"}
    context_opp = {}
    for seq in sequences.values():
        for i in range(1, len(seq) - 1):
            left  = seq[i - 1]
            ref   = seq[i]
            right = seq[i + 1]
            if left not in "ACGT" or ref not in "ACGT" or right not in "ACGT":
                continue
            if ref in ["A", "G"]:
                left, right = comp[right], comp[left]
                ref = comp[ref]
            key = f"{left}[{ref}]{right}"
            context_opp[key] = context_opp.get(key, 0) + 1
    return context_opp


def generate_trinucleotide_counts(vcf_file, fasta_file):
    """
    Count observed mutations in each of the 96 trinucleotide contexts.
    Uses the reference FASTA to look up flanking bases at each mutation site.
    """
    ref_genome     = load_fasta(fasta_file)
    comp           = {"A": "T", "T": "A", "C": "G", "G": "C"}
    context_counts = build_96_context_dictionary()
    skipped        = 0

    with open(vcf_file, "r") as f:
        for line in f:
            if line.startswith("#"):
                continue
            row   = line.strip().split("\t")
            chrom = row[0]
            pos   = int(row[1])
            ref   = row[3].upper()
            alt   = row[4].upper()

            if "," in alt or len(ref) != 1 or len(alt) != 1 or ref == alt:
                skipped += 1
                continue
            if chrom not in ref_genome:
                skipped += 1
                continue

            seq = ref_genome[chrom]
            if pos < 2 or pos > len(seq) - 1:
                skipped += 1
                continue

            left   = seq[pos - 2]
            middle = seq[pos - 1]
            right  = seq[pos]

            if middle != ref or left not in "ACGT" or right not in "ACGT":
                skipped += 1
                continue

            if ref in ["A", "G"]:
                left, right = comp[right], comp[left]
                ref = comp[ref]
                alt = comp[alt]

            key = f"{left}[{ref}>{alt}]{right}"
            if key in context_counts:
                context_counts[key] += 1
            else:
                skipped += 1

    return context_counts, skipped


def normalise_trinucleotide_spectrum(context_counts, context_opp):
    """Normalise observed counts by reference trinucleotide opportunities."""
    normalised = {}
    for key, count in context_counts.items():
        left    = key[0]
        ref     = key[2]
        right   = key[6]
        opp_key = f"{left}[{ref}]{right}"
        opp     = context_opp.get(opp_key, 0)
        normalised[key] = count / opp if opp > 0 else 0.0
    return normalised


# ── Run trinucleotide analysis for all 14 species ────────────────────────────
trinuc_results = {}

for species in vcf_files:
    fasta_path      = fasta_files[species]
    vcf_path        = vcf_files[species]
    context_opp     = count_reference_trinucleotide_contexts(fasta_path)
    counts, skipped = generate_trinucleotide_counts(vcf_path, fasta_path)
    normalised      = normalise_trinucleotide_spectrum(counts, context_opp)

    trinuc_results[species] = {
        "counts":        counts,
        "opportunities": context_opp,
        "normalised":    normalised,
    }
    print(f"{species}: {sum(counts.values())} mutations classified, {skipped} skipped")


# ── Export to CSV ─────────────────────────────────────────────────────────────
mutation_types = ["C>A", "C>G", "C>T", "T>A", "T>C", "T>G"]
rows = []

for species in vcf_files:
    counts        = trinuc_results[species]["counts"]
    opportunities = trinuc_results[species]["opportunities"]
    total_muts    = sum(counts.values())

    for mut in mutation_types:
        for left in bases:
            for right in bases:
                context = f"{left}[{mut}]{right}"
                opp_key = f"{left}[{mut[0]}]{right}"
                n       = counts.get(context, 0)
                rows.append({
                    "species":    species,
                    "context":    context,
                    "mut_class":  mut,
                    "left":       left,
                    "right":      right,
                    "n_mut":      n,
                    "opp":        opportunities.get(opp_key, 0),
                    "proportion": n / total_muts if total_muts > 0 else 0.0,
                })

with open("trinucleotide_data_14sp.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["species", "context", "mut_class",
                                           "left", "right", "n_mut", "opp", "proportion"])
    writer.writeheader()
    writer.writerows(rows)

print(f"Written {len(rows)} rows to trinucleotide_data_14sp.csv")

B. pertussis: 2248 mutations classified, 0 skipped
C. jejuni: 6149 mutations classified, 0 skipped
E. coli: 32936 mutations classified, 0 skipped
H. influenzae: 4459 mutations classified, 0 skipped
K. pneumoniae: 64506 mutations classified, 0 skipped
L. monocytogenes: 11662 mutations classified, 0 skipped
M. tuberculosis: 31209 mutations classified, 0 skipped
N. meningitidis: 4613 mutations classified, 0 skipped
P. aeruginosa: 17257 mutations classified, 0 skipped
S. typhimurium: 3652 mutations classified, 0 skipped
S. aureus: 21116 mutations classified, 0 skipped
S. epidermidis: 15667 mutations classified, 0 skipped
S. agalactiae: 10595 mutations classified, 0 skipped
S. pneumoniae: 6171 mutations classified, 0 skipped
Written 1344 rows to trinucleotide_data_14sp.csv
